# Fine-tune Pegasus on arXiv (Colab)

Runs the full training pipeline from this repo on a Colab GPU and publishes the model to the Hugging Face Hub,
where the deployed app loads it from.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (or better). Then `Runtime → Run all`.
You will be asked for a Hugging Face **write** token in step 2.

| Step | What happens | Approx. time on a T4 |
|---|---|---|
| Ingest + transform | Stream a subset of `ccdv/arxiv-summarization`, tokenize | ~5 min |
| Train | Fine-tune `google/pegasus-cnn_dailymail` | ~1–1.5 h |
| Evaluate | ROUGE for base vs fine-tuned on held-out papers | ~30 min |
| Publish | Push model + model card to the Hub | ~2 min |

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/ThisLonzoBall/Research-Summarizer.git
%cd Research-Summarizer
!pip install -q -r requirements.txt

## 2. Log in to Hugging Face
Create a token with **write** access at https://huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 3. (Optional) adjust the run size
Defaults in `params.yaml`: 4,000 training papers, 1 epoch, 200 test papers for evaluation.
Lower `train_samples` if your session is short on time.

In [ ]:
import yaml

with open("params.yaml") as f:
    params = yaml.safe_load(f)

# params["data"]["train_samples"] = 2000
# params["TrainingArguments"]["num_train_epochs"] = 1

with open("params.yaml", "w") as f:
    yaml.safe_dump(params, f, sort_keys=False)

print(yaml.safe_dump({"data": params["data"], "TrainingArguments": params["TrainingArguments"]}, sort_keys=False))

## 4. Ingest and tokenize the data

In [ ]:
!python main.py --stage ingest transform

## 5. Fine-tune

In [ ]:
!python main.py --stage train

## 6. Evaluate: base vs fine-tuned ROUGE

In [ ]:
!python main.py --stage eval

import json
import pandas as pd

metrics = json.load(open("artifacts/model_evaluation/metrics.json"))
print(f"ROUGE (F1 x 100) on {metrics['num_samples']} held-out arXiv papers")
pd.DataFrame(metrics["scores"]).T

## 7. Publish to the Hugging Face Hub
Set `REPO_ID`, then set the Space's `MODEL_ID` variable to the same value.

In [ ]:
from huggingface_hub import HfApi, ModelCard
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

REPO_ID = "your-hf-username/pegasus-arxiv-summarizer"  # <-- change me
MODEL_DIR = "artifacts/model_trainer/pegasus-arxiv"

AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).push_to_hub(REPO_ID)
AutoTokenizer.from_pretrained(MODEL_DIR).push_to_hub(REPO_ID)

scores = metrics["scores"]
rows = "\n".join(
    f"| {name} | {s['rouge1']} | {s['rouge2']} | {s['rougeL']} | {s['rougeLsum']} |" for name, s in scores.items()
)
card = f"""---
license: mit
base_model: google/pegasus-cnn_dailymail
datasets: [ccdv/arxiv-summarization]
language: [en]
pipeline_tag: summarization
tags: [summarization, pegasus, arxiv, scientific-papers]
---

# Pegasus fine-tuned for arXiv paper summarization

`google/pegasus-cnn_dailymail` fine-tuned on {params["data"]["train_samples"]:,} papers from
[ccdv/arxiv-summarization](https://huggingface.co/datasets/ccdv/arxiv-summarization) to generate abstracts
from paper bodies. Trained with the pipeline in
[Research-Summarizer](https://github.com/ThisLonzoBall/Research-Summarizer).

## Evaluation

ROUGE F1 (x100) on {metrics["num_samples"]} held-out test papers:

| Model | ROUGE-1 | ROUGE-2 | ROUGE-L | ROUGE-Lsum |
|---|---|---|---|---|
{rows}

## Limitations

Inputs are truncated to 1,024 tokens per pass; the app summarizes longer papers chunk by chunk.
"""
ModelCard(card).push_to_hub(REPO_ID)
HfApi().upload_file(
    path_or_fileobj="artifacts/model_evaluation/metrics.json", path_in_repo="metrics.json", repo_id=REPO_ID
)
print(f"https://huggingface.co/{REPO_ID}")

Copy the ROUGE table above into the project README (or paste it back to your assistant).